# Preprocessing Dataset Transfermarkt

Notebook ini menjalankan Task 2 saja: cleaning, filter scope dataset, label target, historical features yang valid, dan penyimpanan dataset preprocessing.

Input:
- `data/raw/players_raw.csv`

Output:
- `data/processed/transfermarkt_dataset_clean.csv`
- `data/model/players_model.csv`


## 1. Import dan Path

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing.clean_dataset import (
    RAW_PLAYERS_FILE,
    PROCESSED_FILE,
    MODEL_FILE,
    build_preprocessed_dataset,
    save_outputs,
)

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw input   : {RAW_PLAYERS_FILE}')
print(f'Clean output: {PROCESSED_FILE}')
print(f'Model output: {MODEL_FILE}')

Project root: c:\KULIAH\SEMESTER 6\BIG DATA\UAS\FINAL
Raw input   : C:\KULIAH\SEMESTER 6\BIG DATA\UAS\FINAL\data\raw\players_raw.csv
Clean output: C:\KULIAH\SEMESTER 6\BIG DATA\UAS\FINAL\data\processed\transfermarkt_dataset_clean.csv
Model output: C:\KULIAH\SEMESTER 6\BIG DATA\UAS\FINAL\data\model\players_model.csv


## 2. Jalankan Preprocessing

In [2]:
clean_df, model_df, report = build_preprocessed_dataset(RAW_PLAYERS_FILE)
processed_file, model_file = save_outputs(clean_df, model_df)

print('Preprocessing selesai.')
print(f'Processed file: {processed_file}')
print(f'Model file    : {model_file}')
print(f'Raw rows      : {report["raw_rows"]}')
print(f'Clean rows    : {report["clean_rows"]}')
print(f'Model rows    : {report["model_rows"]}')
print(f'Duplicate subset: {report["duplicate_subset"]}')
print(f'Duplicates removed before history: {report["duplicates_removed_before_history"]}')

Preprocessing selesai.
Processed file: C:\KULIAH\SEMESTER 6\BIG DATA\UAS\FINAL\data\processed\transfermarkt_dataset_clean.csv
Model file    : C:\KULIAH\SEMESTER 6\BIG DATA\UAS\FINAL\data\model\players_model.csv
Raw rows      : 30024
Clean rows    : 10211
Model rows    : 10211
Duplicate subset: ['player_id', 'season']
Duplicates removed before history: 1593


## 3. Validasi Scope dan Label

In [3]:
print('Shape clean:', clean_df.shape)
print('Shape model:', model_df.shape)
print('Seasons:', sorted(clean_df['season'].unique()))
print('Leagues:', clean_df['league'].value_counts().to_dict())
print('Label distribution:')
print(clean_df['market_value_category'].value_counts())

assert clean_df['season'].between(2017, 2024).all()
assert (clean_df['market_value_mio'] >= 5).all()
assert set(clean_df['market_value_category'].unique()) <= {'Rendah', 'Menengah', 'Tinggi'}
assert not clean_df.duplicated(['player_id', 'season']).any()

print('Validasi scope dan label lulus.')

Shape clean: (10211, 45)
Shape model: (10211, 36)
Seasons: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Leagues: {'Premier League': 3134, 'Serie A': 1926, 'La Liga': 1767, 'Bundesliga': 1753, 'Ligue 1': 1631}
Label distribution:
market_value_category
Menengah    4858
Rendah      3727
Tinggi      1626
Name: count, dtype: int64
Validasi scope dan label lulus.


## 4. Validasi Leakage untuk Dataset Model

In [4]:
forbidden_feature_columns = {
    'market_value_mio',
    'market_value_str',
    'value_category',
    'label',
    'target',
    'mv_growth_rate',
    'position_detail',
}

leakage_columns = sorted((set(model_df.columns) - {'market_value_category'}) & forbidden_feature_columns)
assert leakage_columns == [], f'Leakage columns found: {leakage_columns}'
assert 'position_detail' not in model_df.columns, 'position_detail tidak boleh masuk dataset model'

print('Kolom model:')
print(list(model_df.columns))
print('Validasi leakage lulus.')


Kolom model:
['age', 'age_squared', 'age_group', 'age_peak_distance', 'is_peak_age', 'height_m', 'preferred_foot', 'pos_category', 'is_goalkeeper', 'is_defender', 'is_midfielder', 'is_forward', 'nationality', 'club', 'league', 'league_rank', 'season', 'club_total_mv_mio', 'club_total_mv_log', 'club_total_mv_rank_league_season', 'club_total_mv_pct_league_season', 'prev_season_mv', 'prev_season_mv_log', 'prev_mv_category', 'prev_mv_distance_to_10', 'prev_mv_distance_to_30', 'two_seasons_ago_mv', 'two_seasons_ago_mv_log', 'two_seasons_ago_mv_category', 'has_prev_mv', 'mv_history_count', 'prev_growth_rate', 'prev_growth_rate_clipped', 'prev_mv_to_club_total_ratio', 'age_prev_mv_interaction', 'market_value_category']
Validasi leakage lulus.


## 5. Catatan Kualitas Data

In [5]:
print('Null clean dataset:')
print(clean_df.isna().sum().sort_values(ascending=False))
print()
print('Distribusi position_detail di clean dataset:')
print(clean_df['position_detail'].value_counts(dropna=False).head(10))
print()
print('Catatan: position_detail null 100 persen di raw data, sehingga tidak digunakan sebagai fitur model. Fitur posisi utama adalah pos_category.')


Null clean dataset:
date_of_birth                       7
player_id                           0
club_total_mv_rank_league_season    0
is_goalkeeper                       0
is_defender                         0
is_midfielder                       0
is_forward                          0
age_squared                         0
age_group                           0
age_peak_distance                   0
is_peak_age                         0
club_total_mv_log                   0
club_total_mv_pct_league_season     0
prev_growth_rate                    0
prev_season_mv_log                  0
two_seasons_ago_mv_log              0
prev_mv_category                    0
two_seasons_ago_mv_category         0
prev_mv_distance_to_10              0
prev_mv_distance_to_30              0
prev_growth_rate_clipped            0
prev_mv_to_club_total_ratio         0
market_value_category               0
mv_history_count                    0
player_name                         0
club                          